# Amazon Data Preprocessing

Questo notebook esegue il parsing di amazon-meta.txt e genera i file nodes.csv ed edges_ids.txt per l'analisi successiva.

In [4]:
# 1. Mappatura ASIN -> ID e creazione edges_ids.txt
asin_to_id = {}
all_ids = set()

with open('data/amazon-meta.txt') as f:
    node_id = None
    asin = None
    for line in f:
        if line.startswith('Id:'):
            node_id = line.split()[1]
            all_ids.add(node_id)
        elif line.startswith('ASIN:'):
            asin = line.split()[1]
        if node_id and asin:
            asin_to_id[asin] = node_id
            node_id = None
            asin = None

# Creazione del file edges_ids.txt, sostituendo ASIN con ID, questo file contiene gli archi dei prodotti simili

with open('data/amazon-meta.txt') as f, open('data/edges_ids.txt', 'w') as out:
    node_id = None
    for line in f:
        if line.startswith('Id:'):
            node_id = line.split()[1]
        elif line.startswith('  similar:') and node_id is not None:
            parts = line.strip().split()
            for asin in parts[2:]:
                dst_id = asin_to_id.get(asin)
                if dst_id:
                    out.write(f"{node_id}\t{dst_id}\n")
print(f"Totale nodi: {len(all_ids)}")
print(f"Mappature ASIN: {len(asin_to_id)}")

Totale nodi: 548552
Mappature ASIN: 548552


In [5]:
# 2. Estrazione attributi nodi in nodes.csv
import csv

with open('data/amazon-meta.txt') as fin, open('data/nodes.csv', 'w', newline='', encoding='utf-8') as fout:
    writer = csv.writer(fout)
    writer.writerow(['id', 'asin', 'title', 'group', 'salesrank'])
    node = {}
    for line in fin:
        if line.startswith('Id:'):
            node['id'] = line.split('Id:')[1].strip()
        elif line.startswith('ASIN:'):
            node['asin'] = line.split('ASIN:')[1].strip()
        elif line.startswith('  title:'):
            node['title'] = line.split('  title:')[1].strip()
        elif line.startswith('  group:'):
            node['group'] = line.split('  group:')[1].strip()
        elif line.startswith('  salesrank:'):
            node['salesrank'] = line.split('  salesrank:')[1].strip()
        elif line.strip() == "":
            if 'id' in node and 'asin' in node:
                writer.writerow([
                    node.get('id', ''),
                    node.get('asin', ''),
                    node.get('title', ''),
                    node.get('group', ''),
                    node.get('salesrank', '')
                ])
            node = {}
print("Attributi nodi salvati in nodes.csv")

Attributi nodi salvati in nodes.csv


In [6]:
# 3. Estrazione delle recensioni in ratings.csv

# Creazione del file ratings.csv che servirà per le reccomendations
import csv

with open('data/amazon-meta.txt') as f, open('data/ratings.csv', 'w', newline='', encoding='utf-8') as out:
    writer = csv.writer(out)
    writer.writerow(['user_id', 'product_id', 'rating', 'helpful'])
    current_product_id = None
    review_count = 0
    for line in f:
        if line.startswith('Id:'):
            current_product_id = line.split()[1]
        # Nota: il file ha il typo 'cutomer' invece di 'customer'
        elif 'cutomer:' in line and current_product_id:
            parts = line.strip().split()
            try:
                cutomer_idx = parts.index('cutomer:')
                rating_idx = parts.index('rating:')
                helpful_idx = parts.index('helpful:')
                user_id = parts[cutomer_idx + 1]
                rating = parts[rating_idx + 1]
                helpful = parts[helpful_idx + 1]
                writer.writerow([user_id, current_product_id, rating, helpful])
                review_count += 1
            except (ValueError, IndexError):
                continue  # Salta righe malformate
print(f"Recensioni estratte in ratings.csv: {review_count:,}")

Recensioni estratte in ratings.csv: 7,593,244
